# M4.3 Charged Particles in E and B: Boris versus RK4

Book pointer: chapter 11, section 11.2 (magnetic forces).

## 1. Recall

1. Write the Lorentz force on a charge $q$ moving with velocity $\mathbf{v}$ in fields $\mathbf{E}$ and $\mathbf{B}$. Which term can change the speed, and why can the other not?
2. A charge moves perpendicular to a uniform $\mathbf{B}$. Derive the radius of its circle and the angular frequency, and say which of the two depends on the speed.
3. Add a uniform $\mathbf{E}$ perpendicular to $\mathbf{B}$. In which direction does the guiding centre drift, how fast, and why does the answer not contain $q$ or $m$?
4. A particle moves along a field line into a region of stronger $|\mathbf{B}|$. Which quantity stays constant and what happens to the parallel velocity?
5. You integrate the motion for a million gyro-periods. What property of the exact motion should the integrator keep, and what goes wrong if it does not?

## 2. Principle to algorithm

The equation of motion $m\,d\mathbf{v}/dt = q(\mathbf{E} + \mathbf{v}\times\mathbf{B})$, $d\mathbf{x}/dt = \mathbf{v}$ is a first-order system in the six numbers $(\mathbf{x}, \mathbf{v})$, and any general-purpose integrator will advance it. Classical fourth-order Runge-Kutta (`integrators.rk4`) evaluates the acceleration four times per step and has an error per step of order $\Delta t^5$. It is accurate for short runs and is the right tool when the force has no special structure.

The magnetic force has special structure: it is always perpendicular to $\mathbf{v}$, so it rotates the velocity without changing its length. The Boris pusher (`integrators.boris`) is built around that fact. One step is three operations. A half kick from the electric field, $\mathbf{v}^- = \mathbf{v} + \tfrac{q\mathbf{E}}{m}\tfrac{\Delta t}{2}$. A rotation of $\mathbf{v}^-$ about $\mathbf{B}$ by the angle the particle turns in one step, done with two cross products: with $\mathbf{t} = \tfrac{q\mathbf{B}}{m}\tfrac{\Delta t}{2}$ and $\mathbf{s} = 2\mathbf{t}/(1 + t^2)$, set $\mathbf{v}' = \mathbf{v}^- + \mathbf{v}^-\times\mathbf{t}$ and $\mathbf{v}^+ = \mathbf{v}^- + \mathbf{v}'\times\mathbf{s}$. The vector $\mathbf{t}$ is the half-angle tangent of the rotation and the factor in $\mathbf{s}$ is exactly what makes $|\mathbf{v}^+| = |\mathbf{v}^-|$ for any step size. Then the second half kick, and the position moves with the new velocity, $\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{v}_{k+1}\Delta t$.

Boris is only second order per step, so for a few periods RK4 is more accurate. Over many periods the picture reverses. RK4 has no reason to conserve the speed in a pure magnetic field, and its per-step error, however small, has a systematic sign that accumulates. Boris conserves the speed exactly at every step, so its energy error is rounding error for any run length; its remaining error is a small phase drift. Both routines return arrays `x` and `v` of shape `(n_steps + 1, 3)`. In `boris`, `v[k]` is the velocity half a step before `x[k]`, as in any leapfrog scheme; that offset is first order in $\Delta t$ and is invisible at the step sizes used here except as a slight shift of a circular orbit's centre.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import integrators as it
from emlib import biot_savart as bs

plt.rcParams["figure.figsize"] = (6, 5)

q, m = 1.602e-19, 1.673e-27          # a proton
B0 = 0.01                            # 10 mT along z
omega = it.cyclotron_frequency(q, B0, m)
T = 2 * np.pi / omega
print(f"cyclotron frequency {omega:.4e} rad/s, period {T:.4e} s")

### A circle in a uniform field

Launch the proton across the field at $v_0 = 10^5\,$m/s. It should trace a circle of radius $r_L = mv_0/qB$ once per period $T = 2\pi m/qB$. The centre is taken as the mean position over one period (the leapfrog stagger shifts it slightly from the naive guess $-r_L\hat{\mathbf{y}}$), the radius is the distance from that centre, and the period is read from the accumulated phase angle.

In [ ]:
v0 = 1e5
r_L = it.larmor_radius(m, v0, q, B0)
steps_per_period = 400
x, v = it.boris([0, 0, 0], [v0, 0, 0], q, m, [0, 0, 0], [0, 0, B0], T / steps_per_period, 3 * steps_per_period)

centre = x[:steps_per_period].mean(axis=0)
radius = np.linalg.norm(x - centre, axis=1)
phase = np.unwrap(np.arctan2(x[:, 1] - centre[1], x[:, 0] - centre[0]))
T_measured = 2 * np.pi * (3 * T) / abs(phase[-1] - phase[0])
print(f"Larmor radius  {r_L:.5f} m   measured {radius.mean():.5f} m   relative error {abs(radius.mean() / r_L - 1):.1e}   spread {radius.std() / r_L:.1e}")
print(f"period         {T:.5e} s   measured {T_measured:.5e} s   relative error {abs(T_measured / T - 1):.1e}")
print(f"the orbit turns {'clockwise' if centre[1] < 0 else 'counter-clockwise'} seen from +z, as q v x B demands for a positive charge")

plt.plot(x[:, 0], x[:, 1], lw=1)
plt.plot(*centre[:2], "k+", ms=10); plt.plot(x[0, 0], x[0, 1], "o", color="C3", label="start")
plt.gca().set_aspect("equal"); plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.legend(); plt.title("proton in 10 mT, three periods"); plt.show()

### A helix

Add a velocity component along $\mathbf{B}$. The magnetic force never touches it, so the circle becomes a helix whose pitch is $v_\parallel T$.

In [ ]:
v_par = 3e4
xh, vh = it.boris([0, 0, 0], [v0, 0, v_par], q, m, [0, 0, 0], [0, 0, B0], T / steps_per_period, 4 * steps_per_period)
pitch = xh[steps_per_period, 2] - xh[0, 2]
print(f"advance along B per period {pitch:.5f} m   v_par T = {v_par * T:.5f} m   v_z stays {vh[:, 2].min():.1f} to {vh[:, 2].max():.1f} m/s")

ax = plt.figure(figsize=(6, 5)).add_subplot(projection="3d")
ax.plot(xh[:, 0], xh[:, 1], xh[:, 2], lw=1)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z"); ax.set_title("helix: v_parallel is untouched by B"); plt.show()

### Crossed E and B: the drift

With $\mathbf{E} = E_0\hat{\mathbf{x}}$ and $\mathbf{B} = B_0\hat{\mathbf{z}}$ a particle starting at rest is pulled along $\mathbf{E}$, turned by $\mathbf{B}$, slowed by $\mathbf{E}$ on the way back, and ends each cycle at rest again, displaced sideways. The path is a cycloid and the average velocity is the drift $\mathbf{v}_d = \mathbf{E}\times\mathbf{B}/B^2$, here $-(E_0/B_0)\hat{\mathbf{y}}$. Neither $q$ nor $m$ appears: a drifting plasma carries electrons and ions the same way, which is why this drift produces no current.

In [ ]:
E0 = 100.0
n_per = 200; periods = 10
xd, vd = it.boris([0, 0, 0], [0, 0, 0], q, m, [E0, 0, 0], [0, 0, B0], T / n_per, n_per * periods)
drift = (xd[-1] - xd[0]) / (periods * T)
print(f"measured drift velocity {drift} m/s")
print(f"E x B / B^2            [0, {-E0 / B0:.1f}, 0] m/s   relative error in v_y {abs(drift[1] / (-E0 / B0) - 1):.1e}")
print(f"largest speed reached {np.linalg.norm(vd, axis=1).max():.1f} m/s = 2 E/B = {2 * E0 / B0:.1f} m/s, at the top of each arch")

plt.plot(xd[:, 1], xd[:, 0], lw=1)
plt.gca().set_aspect("equal"); plt.xlabel("y [m]   (drift direction)"); plt.ylabel("x [m]   (E direction)"); plt.title("cycloid of a proton starting at rest in crossed E and B"); plt.show()

### Energy over a long run

Now a pure magnetic field, a coarse step of twelve per period, and 500 periods. The exact motion has constant speed. Boris keeps it to rounding error. RK4 loses energy steadily: a few percent after ten periods and most of it by the end. The RK4 error per step is tiny, but it always has the same sign.

In [ ]:
dt_coarse = T / 12
n_long = 12 * 500
_, vb = it.boris([0, 0, 0], [v0, 0, 0], q, m, [0, 0, 0], [0, 0, B0], dt_coarse, n_long)
_, vr = it.rk4([0, 0, 0], [v0, 0, 0], q, m, [0, 0, 0], [0, 0, B0], dt_coarse, n_long)
ke_b = np.sum(vb**2, axis=1) / v0**2
ke_r = np.sum(vr**2, axis=1) / v0**2
t_periods = np.arange(n_long + 1) / 12
print(f"Boris: max |KE/KE0 - 1| over 500 periods = {np.abs(ke_b - 1).max():.1e}")
print(f"RK4:   KE/KE0 - 1 after 10 periods = {ke_r[120] - 1:+.4f}, after 500 periods = {ke_r[-1] - 1:+.4f}")

plt.semilogy(t_periods, np.abs(ke_b - 1) + 1e-16, label="Boris")
plt.semilogy(t_periods, np.abs(ke_r - 1) + 1e-16, label="RK4")
plt.xlabel("gyro-periods"); plt.ylabel("|KE / KE0 - 1|"); plt.legend(); plt.title("energy error at 12 steps per period"); plt.show()

### Exercise: write one Boris step yourself

Replace the body of `my_boris_step` with your own half kick, rotation, half kick and position update for uniform fields given as arrays. The check compares one step of yours with one step of `integrators.boris` from the same state.

In [ ]:
def my_boris_step(x, v, q, m, E, B, dt):
    """One Boris step for constant E and B (arrays of length 3). Returns (x_new, v_new)."""
    x, v, E, B = (np.asarray(a, dtype=float) for a in (x, v, E, B))     # replace from here with your own version
    v_minus = v + 0.5 * (q / m) * E * dt
    t = 0.5 * (q / m) * B * dt
    s = 2 * t / (1 + t @ t)
    v_prime = v_minus + np.cross(v_minus, t)
    v_plus = v_minus + np.cross(v_prime, s)
    v_new = v_plus + 0.5 * (q / m) * E * dt
    return x + v_new * dt, v_new


x_mine, v_mine = my_boris_step([0.01, 0.02, 0.0], [v0, 2e4, 1e4], q, m, [50.0, 0, 20.0], [0.002, 0, B0], T / 20)
x_lib, v_lib = it.boris([0.01, 0.02, 0.0], [v0, 2e4, 1e4], q, m, [50.0, 0, 20.0], [0.002, 0, B0], T / 20, 1)
print("position difference", np.abs(x_mine - x_lib[1]).max(), "  velocity difference", np.abs(v_mine - v_lib[1]).max())
assert np.allclose(x_mine, x_lib[1], rtol=1e-12, atol=1e-15) and np.allclose(v_mine, v_lib[1], rtol=1e-12, atol=1e-9)
print("one Boris step matches the library")

## 4. Verify

Closed forms with asserts. The Larmor radius is proportional to the speed and the period is independent of it, so a sweep over launch speeds checks both at once. Then the drift velocity and the long-run energy behaviour from the cells above.

In [ ]:
speeds = np.array([2e4, 5e4, 1e5, 2e5, 5e5])
r_meas, T_meas = [], []
for s in speeds:
    xs, _ = it.boris([0, 0, 0], [s, 0, 0], q, m, [0, 0, 0], [0, 0, B0], T / 400, 800)
    c = xs[:400].mean(axis=0)
    r_meas.append(np.linalg.norm(xs - c, axis=1).mean())
    ph = np.unwrap(np.arctan2(xs[:, 1] - c[1], xs[:, 0] - c[0]))
    T_meas.append(2 * np.pi * 2 * T / abs(ph[-1] - ph[0]))
r_meas, T_meas = np.array(r_meas), np.array(T_meas)
r_exact = it.larmor_radius(m, speeds, q, B0)
print("speed [m/s]     r_L [m]   measured   rel err    T measured / T")
for s, re, rm, tm in zip(speeds, r_exact, r_meas, T_meas):
    print(f"{s:9.0f}   {re:9.5f}  {rm:9.5f}  {abs(rm / re - 1):8.1e}   {tm / T:.6f}")
assert np.all(np.abs(r_meas / r_exact - 1) < 1e-3) and np.all(np.abs(T_meas / T - 1) < 1e-3)
assert abs(drift[1] / (-E0 / B0) - 1) < 1e-3 and abs(drift[0]) < 1e-3 * E0 / B0
assert np.abs(ke_b - 1).max() < 1e-10 and abs(ke_r[-1] - 1) > 1e-3
print("radius, period, drift and energy checks passed")

plt.loglog(speeds, r_exact, "k-", label="m v / (q B)")
plt.loglog(speeds, r_meas, "o", color="C1", label="Boris orbit")
plt.xlabel("launch speed [m/s]"); plt.ylabel("orbit radius [m]"); plt.legend(); plt.title("Larmor radius grows with speed, period does not"); plt.show()

## 5. Explore and rebuild

Explore: a magnetic mirror. Two coaxial loops of radius $R = 1\,$m at $z = \pm1\,$m carrying the same current make a field that is weakest between them and strongest in each loop plane. A particle spiralling along the axis towards a loop sees $|\mathbf{B}|$ grow. Its speed cannot change, and its magnetic moment $\mu = mv_\perp^2/2B$ is an adiabatic invariant, so $v_\perp$ grows and $v_\parallel$ shrinks until, if $B$ gets large enough, $v_\parallel$ reaches zero and the particle turns back. The condition is set by the pitch angle $\theta$ between $\mathbf{v}$ and $\mathbf{B}$ at the centre: with $B_{\min}$ at the centre and $B_{\max}$ in the loop plane, the particle reflects if $\sin^2\theta > B_{\min}/B_{\max}$, and escapes through the loss cone otherwise. The field is `biot_savart.wire_B` of 120 segments, called at every step, which is why the loops are coarse and the run short.

The cell launches two protons from near the centre, one inside and one outside the loss cone, and tracks $\mu$ along each path. Then continue on your own: scan the pitch angle in steps of 2 degrees and find the loss-cone angle numerically; compare it with the formula; move the loops apart and see how the mirror ratio and the invariance of $\mu$ change (the adiabatic argument needs the field to change little over one gyro-orbit).

In [ ]:
I_loop, R_loop = 1e5, 1.0
m1, d1 = bs.segments(bs.loop(R_loop, 60, (0, 0, -1.0)))
m2, d2 = bs.segments(bs.loop(R_loop, 60, (0, 0, 1.0)))
mid, dl = np.vstack([m1, m2]), np.vstack([d1, d2])
B_mirror = lambda p: bs.wire_B(mid, dl, I_loop, p)

z_axis = np.linspace(-1.5, 1.5, 61)
Bz_axis = B_mirror(np.stack([0 * z_axis, 0 * z_axis, z_axis], axis=-1))[:, 2]
B_min, B_max = Bz_axis[30], Bz_axis[np.argmin(np.abs(z_axis - 1.0))]
theta_loss = np.degrees(np.arcsin(np.sqrt(B_min / B_max)))
print(f"B on axis: centre {B_min:.4f} T, loop plane {B_max:.4f} T, mirror ratio {B_max / B_min:.3f}, loss-cone angle {theta_loss:.1f} deg")
T_c = 2 * np.pi * m / (q * B_min)
print(f"gyro-period at the centre {T_c:.2e} s, Larmor radius {it.larmor_radius(m, v0, q, B_min):.3f} m (small compared with R = 1 m, so mu should be invariant)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(z_axis, Bz_axis * 1e3, "k"); axes[0].set_xlabel("z [m]"); axes[0].set_ylabel("B_z on axis [mT]"); axes[0].set_title("the mirror field")
for pitch in (70.0, 40.0):
    th = np.radians(pitch)
    xm, vm = it.boris([0.03, 0, 0], [v0 * np.sin(th), 0, v0 * np.cos(th)], q, m, [0, 0, 0], B_mirror, T_c / 20, 3000)
    Bm = B_mirror(xm); Bmag = np.linalg.norm(Bm, axis=1); bhat = Bm / Bmag[:, None]
    v_par = np.sum(vm * bhat, axis=1)
    mu = m * (np.sum(vm**2, axis=1) - v_par**2) / (2 * Bmag)
    fate = "reflects" if np.abs(xm[:, 2]).max() < 1.0 else "escapes"
    print(f"pitch {pitch:.0f} deg ({'inside' if pitch < theta_loss else 'outside'} the loss cone): z range {xm[:, 2].min():+.2f} to {xm[:, 2].max():+.2f} m, {fate};   mu spread / mean = {mu.std() / mu.mean():.1e};   speed constant to {np.abs(np.linalg.norm(vm, axis=1) / v0 - 1).max():.0e}")
    axes[1].plot(xm[:, 2], xm[:, 0], lw=0.8, label=f"pitch {pitch:.0f} deg: {fate}")
axes[1].set_xlim(-1.6, 1.6); axes[1].set_xlabel("z [m]"); axes[1].set_ylabel("x [m]"); axes[1].legend(); axes[1].set_title("two launches from the centre")
plt.tight_layout(); plt.show()

Rebuild. Close this notebook. In a blank editor write the Boris pusher from memory (half kick, rotation with $\mathbf{t}$ and $\mathbf{s}$, half kick, position update) and an RK4 integrator for the same system, then reproduce the energy comparison at twelve steps per period over 500 periods.